# 02 — Data Cleaning (Corrected)

This notebook cleans the **exact datasets supplied for the project**.

### Important decisions
- Supply-chain outliers are reported, not deleted.
- News has no date column, so no artificial date is created.
- Marketing `Duration` and `Acquisition_Cost` are parsed from strings such as `30 days` and `$16,174.00`.
- Historical trade missing values are handled according to the meaning of the fields rather than blindly deleting rows.
- No predictions, model metrics, or research-result claims are produced in this notebook.


## Cell 1 — Imports and Paths

In [43]:
from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path.cwd().parent
RAW = ROOT / "data" / "raw"
INTERIM = ROOT / "data" / "interim"
HIST = RAW / "historical_trade"

INTERIM.mkdir(parents=True, exist_ok=True)

print("ROOT:", ROOT)
print("RAW:", RAW)
print("INTERIM:", INTERIM)


ROOT: c:\Users\ankit\Desktop\Ml-Project\AI-Supply-Chain-Digital-Marketing-v1
RAW: c:\Users\ankit\Desktop\Ml-Project\AI-Supply-Chain-Digital-Marketing-v1\data\raw
INTERIM: c:\Users\ankit\Desktop\Ml-Project\AI-Supply-Chain-Digital-Marketing-v1\data\interim


## Cell 2 — Load the Three Main Supplied Datasets

In [44]:
supply_chain = pd.read_csv(RAW / "supply_chain.csv")

marketing = pd.read_csv(
    RAW / "marketing_campaign.csv"
)

# Supplied news file has no header row.
news = pd.read_csv(
    RAW / "news_sentiment.csv",
    header=None,
    names=["sentiment", "text"],
    encoding="latin1"
)

print("Supply chain:", supply_chain.shape)
print("Marketing:", marketing.shape)
print("News:", news.shape)


Supply chain: (5000, 14)
Marketing: (200000, 16)
News: (4846, 2)


## Cell 3 — Helper Functions

In [46]:
def clean_text_columns(df):
    df = df.copy()

    for col in df.select_dtypes(include=["object", "string"]).columns:
        df[col] = (
            df[col]
            .astype("string")
            .str.strip()
            .str.replace(r"\s+", " ", regex=True)
        )

    return df


def extract_number(series):
    # Converts examples such as:
    # '30 days' -> 30
    # '$16,174.00' -> 16174.00
    # while preserving missing values.
    return pd.to_numeric(
        series.astype("string")
              .str.replace(",", "", regex=False)
              .str.extract(r"([-+]?\d*\.?\d+)", expand=False),
        errors="coerce"
    )


## Cell 4 — Clean Supply Chain

In [47]:
supply_clean = supply_chain.copy()

before = len(supply_clean)

supply_clean = supply_clean.drop_duplicates().reset_index(drop=True)
supply_clean = clean_text_columns(supply_clean)

supply_clean["Date"] = pd.to_datetime(
    supply_clean["Date"],
    errors="coerce"
)

numeric_cols = [
    "Distance_km",
    "Weight_MT",
    "Fuel_Price_Index",
    "Geopolitical_Risk_Score",
    "Carrier_Reliability_Score",
    "Lead_Time_Days",
    "Disruption_Occurred"
]

for col in numeric_cols:
    supply_clean[col] = pd.to_numeric(
        supply_clean[col],
        errors="coerce"
    )

print("Duplicate rows removed:", before - len(supply_clean))
print("Shape after cleaning:", supply_clean.shape)


Duplicate rows removed: 0
Shape after cleaning: (5000, 14)


## Cell 5 — Supply Chain Missing Values and Target Check

In [48]:
print("Supply-chain missing values:")
display(supply_clean.isna().sum().to_frame("missing"))

print("\nTarget distribution:")
display(
    supply_clean["Disruption_Occurred"]
    .value_counts()
    .sort_index()
    .to_frame("count")
)

checks = pd.Series({
    "negative_distance": int((supply_clean["Distance_km"] < 0).sum()),
    "negative_weight": int((supply_clean["Weight_MT"] < 0).sum()),
    "negative_lead_time": int((supply_clean["Lead_Time_Days"] < 0).sum()),
    "target_not_0_or_1": int(
        (~supply_clean["Disruption_Occurred"].isin([0, 1])).sum()
    )
}, name="count")

display(checks.to_frame())


Supply-chain missing values:


,missing
Shipment_ID,0
Date,0
Origin_Port,0
Destination_Port,0
Transport_Mode,0
Product_Category,0
Distance_km,0
Weight_MT,0
Fuel_Price_Index,0
Geopolitical_Risk_Score,0



Target distribution:


,count
Disruption_Occurred,
0,1937
1,3063


,count
negative_distance,0
negative_weight,0
negative_lead_time,0
target_not_0_or_1,0


## Cell 6 — Supply Chain Outlier Report

In [49]:
def outlier_report(df, columns):
    rows = []

    for col in columns:
        x = pd.to_numeric(df[col], errors="coerce").dropna()

        q1 = x.quantile(0.25)
        q3 = x.quantile(0.75)
        iqr = q3 - q1

        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr

        count = int(((x < lower) | (x > upper)).sum())

        rows.append({
            "column": col,
            "q1": q1,
            "q3": q3,
            "lower_bound": lower,
            "upper_bound": upper,
            "outlier_count": count
        })

    return pd.DataFrame(rows)


supply_outliers = outlier_report(
    supply_clean,
    [
        "Distance_km",
        "Weight_MT",
        "Fuel_Price_Index",
        "Geopolitical_Risk_Score",
        "Carrier_Reliability_Score",
        "Lead_Time_Days"
    ]
)

display(supply_outliers)

print(
    "Note: outliers are retained because extreme lead time, "
    "risk and other values may carry disruption information."
)


,column,q1,q3,lower_bound,upper_bound,outlier_count
0,Distance_km,4036.010,11347.4625,-6931.16875,22314.64125,0
1,Weight_MT,124.330,366.9550,-239.60750,730.89250,0
2,Fuel_Price_Index,2.020,3.7100,-0.51500,6.24500,0
3,Geopolitical_Risk_Score,2.600,7.5000,-4.75000,14.85000,0
4,Carrier_Reliability_Score,0.629,0.8790,0.25400,1.25400,0
5,Lead_Time_Days,2.110,21.2075,-26.53625,49.85375,518


Note: outliers are retained because extreme lead time, risk and other values may carry disruption information.


## Cell 7 — Clean News / NLP Input

In [50]:
news_clean = news.copy()

before = len(news_clean)

news_clean = news_clean.drop_duplicates().reset_index(drop=True)
news_clean = clean_text_columns(news_clean)

news_clean["sentiment"] = (
    news_clean["sentiment"]
    .str.lower()
    .str.strip()
)

# Remove records without usable text.
news_clean = news_clean.dropna(subset=["text"]).copy()
news_clean = news_clean[
    news_clean["text"].str.len() > 0
].reset_index(drop=True)

print("News duplicate/empty records removed:", before - len(news_clean))
print("News shape after cleaning:", news_clean.shape)

print("\nNews missing values:")
display(news_clean.isna().sum().to_frame("missing"))

print("\nNews sentiment values:")
display(news_clean["sentiment"].value_counts().to_frame("count"))

print("\nNews date column exists:", "date" in [c.lower() for c in news_clean.columns])


News duplicate/empty records removed: 6
News shape after cleaning: (4840, 2)

News missing values:


,missing
sentiment,0
text,0



News sentiment values:


,count
sentiment,
neutral,2873
positive,1363
negative,604



News date column exists: False


## Cell 8 — Clean Marketing Data Correctly

In [51]:
marketing_clean = marketing.copy()

before = len(marketing_clean)

marketing_clean = marketing_clean.drop_duplicates().reset_index(drop=True)
marketing_clean = clean_text_columns(marketing_clean)

marketing_clean["Date"] = pd.to_datetime(
    marketing_clean["Date"],
    errors="coerce"
)

print("Marketing duplicate rows removed:", before - len(marketing_clean))
print("Marketing shape after cleaning:", marketing_clean.shape)

print("\nRaw Duration examples:")
display(marketing["Duration"].head(10).to_frame())

print("\nRaw Acquisition_Cost examples:")
display(marketing["Acquisition_Cost"].head(10).to_frame())


Marketing duplicate rows removed: 0
Marketing shape after cleaning: (200000, 16)

Raw Duration examples:


,Duration
0,30 days
1,60 days
2,30 days
3,60 days
4,15 days
5,15 days
6,60 days
7,45 days
8,15 days
9,15 days



Raw Acquisition_Cost examples:


,Acquisition_Cost
0,"$16,174.00"
1,"$11,566.00"
2,"$10,200.00"
3,"$12,724.00"
4,"$16,452.00"
5,"$9,716.00"
6,"$11,067.00"
7,"$13,280.00"
8,"$18,066.00"
9,"$13,766.00"


## Cell 9 — Parse Marketing Numeric Fields

In [52]:
# IMPORTANT:
# Duration contains values such as '30 days'.
# Acquisition_Cost contains values such as '$16,174.00'.
# Direct pd.to_numeric(..., errors='coerce') would turn these into NaN.

marketing_clean["Duration"] = extract_number(
    marketing_clean["Duration"]
)

marketing_clean["Acquisition_Cost"] = extract_number(
    marketing_clean["Acquisition_Cost"]
)

other_numeric = [
    "Conversion_Rate",
    "ROI",
    "Clicks",
    "Impressions",
    "Engagement_Score"
]

for col in other_numeric:
    marketing_clean[col] = pd.to_numeric(
        marketing_clean[col],
        errors="coerce"
    )

print("Marketing missing values after safe parsing:")
display(marketing_clean.isna().sum().to_frame("missing"))

print("\nParsed Duration summary:")
display(marketing_clean["Duration"].describe().to_frame("Duration"))

print("\nParsed Acquisition_Cost summary:")
display(marketing_clean["Acquisition_Cost"].describe().to_frame("Acquisition_Cost"))


Marketing missing values after safe parsing:


,missing
Campaign_ID,0
Company,0
Campaign_Type,0
Target_Audience,0
Duration,0
Channel_Used,0
Conversion_Rate,0
Acquisition_Cost,0
ROI,0
Location,0



Parsed Duration summary:


,Duration
count,200000.0
mean,37.503975
std,16.74672
min,15.0
25%,30.0
50%,30.0
75%,45.0
max,60.0



Parsed Acquisition_Cost summary:


,Acquisition_Cost
count,200000.0
mean,12504.39304
std,4337.664545
min,5000.0
25%,8739.75
50%,12496.5
75%,16264.0
max,20000.0


## Cell 10 — Marketing Validation

In [53]:
marketing_checks = pd.Series({
    "negative_duration": int((marketing_clean["Duration"] < 0).sum()),
    "negative_acquisition_cost": int((marketing_clean["Acquisition_Cost"] < 0).sum()),
    "negative_clicks": int((marketing_clean["Clicks"] < 0).sum()),
    "negative_impressions": int((marketing_clean["Impressions"] < 0).sum()),
    "conversion_rate_below_0": int((marketing_clean["Conversion_Rate"] < 0).sum()),
    "conversion_rate_above_1": int((marketing_clean["Conversion_Rate"] > 1).sum()),
    "roi_below_0": int((marketing_clean["ROI"] < 0).sum())
}, name="count")

display(marketing_checks.to_frame())


,count
negative_duration,0
negative_acquisition_cost,0
negative_clicks,0
negative_impressions,0
conversion_rate_below_0,0
conversion_rate_above_1,0
roi_below_0,0


## Cell 11 — Load the Supplied Historical Trade Datasets

In [54]:
historical_files = [
    "commodity_prices_supply_chain.csv",
    "disruption_events.csv",
    "industry_exposure.csv",
    "port_congestion.csv",
    "shipping_rates.csv",
    "tariff_timeline.csv",
    "trade_flows.csv"
]

historical = {}

for file_name in historical_files:
    path = HIST / file_name

    if not path.exists():
        print("WARNING — file not found:", path)
        continue

    df = pd.read_csv(path)
    before = len(df)

    df = df.drop_duplicates().reset_index(drop=True)
    df = clean_text_columns(df)

    historical[file_name] = df

    print(
        f"{file_name}: "
        f"{before:,} -> {len(df):,} rows | "
        f"{len(df.columns)} columns"
    )


commodity_prices_supply_chain.csv: 110,515 -> 110,515 rows | 8 columns
disruption_events.csv: 58 -> 58 rows | 18 columns
industry_exposure.csv: 250 -> 250 rows | 15 columns
port_congestion.csv: 6,260 -> 6,260 rows | 11 columns
shipping_rates.csv: 300 -> 300 rows | 12 columns
tariff_timeline.csv: 71 -> 71 rows | 17 columns
trade_flows.csv: 1,250 -> 1,250 rows | 11 columns


## Cell 12 — Historical Missing-Value Report

In [55]:
missing_report = []

for file_name, df in historical.items():
    for col in df.columns:
        n = int(df[col].isna().sum())

        if n > 0:
            missing_report.append({
                "dataset": file_name,
                "column": col,
                "missing_count": n
            })

historical_missing = pd.DataFrame(missing_report)

if historical_missing.empty:
    print("No historical missing values.")
else:
    display(historical_missing)


,dataset,column,missing_count
0,shipping_rates.csv,bdi_mom_change_pct,1
1,shipping_rates.csv,container_yoy_pct,12
2,trade_flows.csv,yoy_growth_pct,50


## Cell 13 — Handle Historical Missing Values

The observed missing values are concentrated in **change/growth fields**:
- `shipping_rates.csv`: `bdi_mom_change_pct` and `container_yoy_pct`
- `trade_flows.csv`: `yoy_growth_pct`

These fields are unavailable when a previous comparison period does not exist. For the downstream feature pipeline, these initial/unavailable growth-change observations are represented as **0**, while an accompanying indicator records that the original value was missing.

This avoids deleting valid historical rows.


In [56]:
historical_final = {}

for file_name, df in historical.items():
    df = df.copy()

    # Change/growth columns where missing values can occur
    # because a previous comparison period is unavailable.
    growth_change_cols = [
        "bdi_mom_change_pct",
        "container_yoy_pct",
        "yoy_growth_pct"
    ]

    for col in growth_change_cols:
        if col in df.columns:
            missing_flag = f"{col}_was_missing"

            df[missing_flag] = df[col].isna().astype("int8")
            df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)

    # Any remaining numeric missing values:
    # use column median rather than dropping valid rows.
    for col in df.columns:
        if df[col].isna().sum() == 0:
            continue

        if pd.api.types.is_numeric_dtype(df[col]):
            median_value = df[col].median()

            if pd.notna(median_value):
                df[col] = df[col].fillna(median_value)

        else:
            df[col] = df[col].fillna("Unknown")

    historical_final[file_name] = df


## Cell 14 — Verify Historical Cleaning

In [57]:
historical_validation = []

for file_name, df in historical_final.items():
    historical_validation.append({
        "dataset": file_name,
        "rows": len(df),
        "columns": len(df.columns),
        "missing_cells": int(df.isna().sum().sum())
    })

historical_validation = pd.DataFrame(historical_validation)

display(historical_validation)

remaining = historical_validation[
    historical_validation["missing_cells"] > 0
]

if remaining.empty:
    print("✅ All historical datasets have 0 missing cells.")
else:
    print("⚠️ Missing values still remain:")
    display(remaining)


,dataset,rows,columns,missing_cells
0,commodity_prices_supply_chain.csv,110515,8,0
1,disruption_events.csv,58,18,0
2,industry_exposure.csv,250,15,0
3,port_congestion.csv,6260,11,0
4,shipping_rates.csv,300,14,0
5,tariff_timeline.csv,71,17,0
6,trade_flows.csv,1250,12,0


✅ All historical datasets have 0 missing cells.


## Cell 15 — Save Cleaned Datasets

In [58]:
supply_clean.to_csv(
    INTERIM / "cleaned_supply_chain.csv",
    index=False
)

news_clean.to_csv(
    INTERIM / "cleaned_news.csv",
    index=False
)

marketing_clean.to_csv(
    INTERIM / "cleaned_marketing.csv",
    index=False
)

for file_name, df in historical_final.items():
    output_name = f"cleaned_{Path(file_name).stem}.csv"
    df.to_csv(
        INTERIM / output_name,
        index=False
    )

print("Saved cleaned datasets to:")
print(INTERIM)


Saved cleaned datasets to:
c:\Users\ankit\Desktop\Ml-Project\AI-Supply-Chain-Digital-Marketing-v1\data\interim


## Cell 16 — Final Validation

In [59]:
final_rows = []

for path in sorted(INTERIM.glob("cleaned_*.csv")):
    df = pd.read_csv(path)

    final_rows.append({
        "file": path.name,
        "rows": len(df),
        "columns": len(df.columns),
        "missing_cells": int(df.isna().sum().sum())
    })

final_validation = pd.DataFrame(final_rows)

display(final_validation)

if (final_validation["missing_cells"] == 0).all():
    print("\n✅ CLEANING COMPLETE — ALL SAVED CLEANED DATASETS HAVE ZERO MISSING CELLS.")
else:
    print("\n⚠️ Some saved cleaned datasets still contain missing values.")
    display(
        final_validation[
            final_validation["missing_cells"] > 0
        ]
    )


,file,rows,columns,missing_cells
0,cleaned_commodity_prices_supply_chain.csv,110515,8,0
1,cleaned_disruption_events.csv,58,18,0
2,cleaned_industry_exposure.csv,250,15,0
3,cleaned_marketing.csv,200000,16,0
4,cleaned_news.csv,4840,2,0
5,cleaned_port_congestion.csv,6260,11,0
6,cleaned_shipping_rates.csv,300,14,0
7,cleaned_supply_chain.csv,5000,14,0
8,cleaned_tariff_timeline.csv,71,17,0
9,cleaned_trade_flows.csv,1250,12,0



✅ CLEANING COMPLETE — ALL SAVED CLEANED DATASETS HAVE ZERO MISSING CELLS.


# Cleaning Complete

The cleaned datasets are now stored in:

`data/interim/`

### Expected main datasets
- `cleaned_supply_chain.csv` → 5,000 × 14
- `cleaned_news.csv` → 4,840 × 2
- `cleaned_marketing.csv` → 200,000 × 16

Next notebook: **`03_eda.ipynb`**

Do not modify the data further before checking the final validation output.
